## P15: Longest run of different merchants (easy–medium)

A card's transactions are given in time order as a list of merchant names. Fraud analysts want the **length of the longest run of consecutive transactions in which no merchant appears twice**. (A run is a block of transactions next to each other, with no gaps.)

**Input:** `merchants`, a list of strings, e.g. `["Tesco", "Costa", "Tesco", "Boots", "Costa", "Greggs"]` (position 0 is the earliest transaction).
**Output:** one whole number, the length of that longest run. An empty list gives 0.

**Worked example** for `["Tesco", "Costa", "Tesco", "Boots", "Costa", "Greggs"]` (positions 0–5). Start a run at each position and extend it until a merchant repeats:

| Run starts at | Run (before a repeat) | Stops because | Length |
|---|---|---|---|
| 0 | Tesco, Costa | Tesco again at position 2 | 2 |
| 1 | Costa, Tesco, Boots | Costa again at position 4 | 3 |
| 2 | Tesco, Boots, Costa, Greggs | end of the list | **4** |
| 3 | Boots, Costa, Greggs | end of the list | 3 |
| 4 | Costa, Greggs | end of the list | 2 |
| 5 | Greggs | end of the list | 1 |

Longest = max(2, 3, 4, 3, 2, 1) = **4**.

| Input `merchants` | Output | Why |
|---|---|---|
| `["Tesco", "Costa", "Tesco", "Boots", "Costa", "Greggs"]` | 4 | Tesco, Boots, Costa, Greggs (positions 2–5) |
| `["Lidl", "Aldi", "Lidl", "Lidl"]` | 2 | Lidl, Aldi or Aldi, Lidl. The two Lidls in a row can't be in the same run |
| `[]` | 0 | no transactions |

**Constraints:** `0 <= len(merchants) <= 200,000`; up to 500 different merchants.

**Target:** O(n) time. The table above (restart at every position) is O(n²) and too slow for 200,000 items.

**Time target: 10 minutes, blank cell, no notes.** Start time: ____

In [1]:
def longest_distinct_run(merchants: list[str]) -> int:
    last_seen = {}          # merchant -> most recent position
    left = best = 0
    for right, m in enumerate(merchants):
        if m in last_seen and last_seen[m] >= left:   # repeat inside the current run
            left = last_seen[m] + 1
        last_seen[m] = right
        best = max(best, right - left + 1)
    return best

In [2]:
# Tests: all asserts must pass silently.
assert longest_distinct_run(["Tesco", "Costa", "Tesco", "Boots", "Costa", "Greggs"]) == 4
assert longest_distinct_run(["Lidl", "Aldi", "Lidl", "Lidl"]) == 2      # repeat right next to itself
assert longest_distinct_run([]) == 0                                     # empty
assert longest_distinct_run(["Tesco"]) == 1                              # single transaction
assert longest_distinct_run(["Costa", "Costa", "Costa"]) == 1            # all the same
assert longest_distinct_run(["A", "B", "C", "D", "E"]) == 5              # no repeats at all
assert longest_distinct_run(["A", "B", "B", "A"]) == 2                   # the old "A" is outside the current run
assert longest_distinct_run(["X", "Y", "Z", "Y", "X", "W"]) == 4         # Z, Y, X, W
print("all tests pass")

all tests pass


## Explain it (2–3 lines each)

1. Approach: what do you keep track of as you go, and what happens when a merchant repeats?
2. Time **and** space complexity, biggest term only, with a reason specific to this algorithm.
3. Why is `["A", "B", "B", "A"]` a trap? What would a wrong version return, and why?

✅ **Model answer**

1. A sliding window over the list. Keep `last_seen[merchant] = position` and a `left` edge. When a merchant repeats **inside** the current run, move `left` to just past its previous position; the run length is `right − left + 1`, and keep the best.
2. **O(n) time**: `right` visits each transaction once and `left` only moves forwards. **O(min(n, k)) space**, where k = the number of different merchants: the dict holds one entry per distinct merchant.
3. At the final `"A"`, `last_seen["A"] = 0` is **outside** the current run (which starts at 2, after the second `"B"`). A version without the `>= left` check jumps `left` **back** to 1 and returns 3 (the run `B, B, A`, which contains two `B`s). The right answer is 2.